In [270]:
# Mahjong CONSTS
from mahjong.constants import EAST, SOUTH, WEST, NORTH, HAKU, HATSU, CHUN #former 4 are .WINDS, and all 7 are .HONOR_INDICES
from mahjong.constants import FIVE_RED_MAN, FIVE_RED_PIN, FIVE_RED_SOU # Red Dora number cards > .AKA_DORA_LIST
from mahjong.constants import DISPLAY_WINDS # Str display of the winds

# Mahjong methods/classes
from mahjong.hand_calculating.hand import HandCalculator
from mahjong.hand_calculating.hand_config import HandConfig, OptionalRules
from mahjong.shanten import Shanten
from mahjong.meld import Meld
from mahjong.tile import TilesConverter
from mahjong.agari import Agari

# imports
import ipywidgets as widgets
from dataclasses import dataclass, field, asdict
from enum import Enum, auto
from typing import List, Optional, Tuple, Union, Set, Dict, Any
from IPython.display import display, HTML, clear_output

import random
import re
import copy
import os
from pathlib import Path
import base64

# Import custom mahjong classes
import helper.config as config
from helper.game import MahjongGame

In [271]:
# VISUALIZE SIMULATION

class MahjongReplay:
    def __init__(self, history):
        self.history = history
        self.current_idx = 0
        self.output = widgets.Output()
        # Find all indices where a new round starts
        self.round_start_indices = self._get_round_indices()

    def _get_round_indices(self):
        indices = [i for i,n in enumerate(self.history) if n["action"] == "SETUP ROUND"]
        return indices

    def render(self):
        if not self.history:
            with self.output:
                print("Error: History is empty.")
            return

        state = self.history[self.current_idx]
        with self.output:
            clear_output(wait=True)
            
            # 1. Logic for Header (Wind Phase and Kyoku)
            # num_wind: 0=East Phase, 1=South Phase
            # num_kyoku: 0 to 3
            field_wind = MahjongTable.BA[state['num_wind']]
            kyoku_display_num = state['num_kyoku'] + 1
            
            # Map Dora
            dora_html = ""
            for d_val in state["dora"]:
                d_val: Hand136
                d_path_list = [n for n in mspzd_to_file_path(d_val.to_mspzd().notation) if len(n)>0]
                d_path = d_path_list[0][0] if d_path_list and d_path_list[0] else None
                if d_path:
                    dora_html += f'<img src="{d_path}" style="height: 40px; border: 1px solid #ffd700; border-radius: 2px; margin-right: 2px;">'

            html = f"""
            <div style="background-color: #1e1e1e; color: #d4d4d4; padding: 20px; border-radius: 10px; font-family: 'Consolas', monospace; border: 1px solid #333; width: fit-content; min-width: 680px;">
                <div style="border-bottom: 2px solid #4ec9b0; padding-bottom: 8px; margin-bottom: 15px; display: flex; justify-content: space-between; align-items: center;">
                    <div>
                        <span style="font-size: 1.2em; font-weight: bold; color: #4ec9b0;">{field_wind} {kyoku_display_num} Kyoku</span>
                        <span style="margin-left: 20px; color: #858585;">Tiles: {state['tiles_remaining']}</span>
                    </div>
                    <div style="display: flex; align-items: center;">
                        <span style="font-size: 0.8em; color: #ffd700; margin-right: 8px;">DORA:</span>
                        {dora_html if dora_html else '<span style="color:#444">None</span>'}
                    </div>
                </div>
            """
            
            # Handle Players
            for i, p in enumerate(state['players']):
                is_active = (i == state['turn_index'])
                active_css = "background: #252526; border: 1px solid #569cd6;" if is_active else "border: 1px solid #333;"
                
                # 1. Convert the base hand (13 tiles) to paths
                # Use p['hand'] directly since it doesn't contain the draw yet
                hand_str = MSPZD(MahjongConverter.to_str(p['hand'])).notation
                path_groups = mspzd_to_file_path(hand_str)
                all_paths = [path for group in path_groups for path in group]
                
                tile_html = ""
                # Render the base hand (typically 13 tiles)
                for path in all_paths:
                    tile_html += f'<img src="{path}" style="height: 60px; width: 38px; margin-left: 1px; border-radius: 4px; border: 1px solid #000; vertical-align: bottom;">'

                # 2. Explicitly render the Drawn tile from state['action_tile']
                if is_active and state.get('action') == "DRAW" and state.get('action_tile'):
                    draw_tile_obj = state['action_tile']
                    draw_tile_obj: MSPZD
                    draw_path_list = [n for n in mspzd_to_file_path(draw_tile_obj.notation) if len(n)>0]
                    if draw_path_list and draw_path_list[0]:
                        draw_path = draw_path_list[0][0]
                        # Add the 15px gap and a gold border to highlight the new arrival
                        tile_html += f'<img src="{draw_path}" style="height: 60px; width: 38px; margin-left: 15px; border-radius: 4px; border: 2px solid #ffd700; vertical-align: bottom;">'

                seat_wind = MahjongTable.BA[p['wind']]

                html += f"""
                <div style="padding: 12px; margin: 10px 0; border-radius: 6px; {active_css}">
                    <div style="display: flex; justify-content: space-between; margin-bottom: 8px;">
                        <span>
                            <b style="color: #9cdcfe;">[{seat_wind}]</b> {p['name']}
                            {'<span style="color: #f44747; font-size: 0.75em; font-weight: bold; margin-left: 10px; border: 1px solid #f44747; padding: 1px 4px; border-radius: 3px;">DEALER</span>' if p['is_dealer'] else ''}
                        </span>
                        <span style="color: #b5cea8; font-weight: bold;">{p['score']:,}</span>
                    </div>
                    <div style="display: flex; align-items: center; background: #111; padding: 10px; border-radius: 4px; min-height: 62px;">
                        {tile_html if tile_html else '<span style="color:#444;">No tiles</span>'}
                    </div>
                </div>
                """
            
            html += f"""
                <div style="margin-top: 10px; font-size: 0.9em; color: #ce9178; background: #2d2d2d; padding: 5px 10px; border-radius: 4px;">
                    Last Action: <b>{state['action']}</b> | Step: {self.current_idx}
                </div>
            </div>
            """
            display(HTML(html))

    # --- Navigation Logic ---
    def move_next_turn(self, _):
        if self.current_idx < len(self.history) - 1:
            self.current_idx += 1
            self.render()

    def move_prev_turn(self, _):
        if self.current_idx > 0:
            self.current_idx -= 1
            self.render()

    def move_next_round(self, _):
        for idx in self.round_start_indices:
            if idx > self.current_idx:
                self.current_idx = idx
                self.render()
                break

    def move_prev_round(self, _):
        for idx in reversed(self.round_start_indices):
            if idx < self.current_idx:
                self.current_idx = idx
                self.render()
                break

    def show(self):
        """Build and display the entire UI."""
        # Create buttons
        b1 = widgets.Button(description="PREV R", layout=widgets.Layout(width='80px'))
        b2 = widgets.Button(description="PREV T", layout=widgets.Layout(width='80px'))
        b3 = widgets.Button(description="NEXT T", layout=widgets.Layout(width='80px'))
        b4 = widgets.Button(description="NEXT R", layout=widgets.Layout(width='80px'))

        # Navigation logic (direct functions)
        def p_r(_): 
            for idx in reversed(self.round_start_indices):
                if idx < self.current_idx:
                    self.current_idx = idx; break
            self.render()
        
        def p_t(_): 
            self.current_idx = max(0, self.current_idx - 1); self.render()
            
        def n_t(_): 
            self.current_idx = min(len(self.history)-1, self.current_idx + 1); self.render()
            
        def n_r(_):
            for idx in self.round_start_indices:
                if idx > self.current_idx:
                    self.current_idx = idx; break
            self.render()

        b1.on_click(p_r); b2.on_click(p_t); b3.on_click(n_t); b4.on_click(n_r)

        # Trigger initial render
        self.render()
        
        # Display everything directly
        ui = widgets.VBox([widgets.HBox([b1, b2, b3, b4]), self.output])
        display(ui)

def mspzd_to_file_path(mspzd: str):
    result = {char: [] for char in "mpszd"}
    buffer = []

    for char in mspzd:
        if char.isdigit():
            buffer.append(char)
        elif char in result:
            result[char].extend([int(d) for d in buffer])
            result[char].sort()
            buffer = []

    # Build the 2D array in fixed order: m, p, s, z, d
    ref = "mpszd"
    paths_2d = []
    for char in ref:
        suit_paths = [
            os.path.join(TILE_IMAGE_DIR, char, f"{char}{num}.png") 
            for num in result[char]
        ]
        paths_2d.append(suit_paths)
            
    return paths_2d

In [272]:
game = MahjongGame(
    player_names = ["lynn", "byron", "fu", "yagata"],
    starting_score = 25000,
    total_ba = 2,
    enable_red_dora = True,
    log_level = "full"
)

results = game.play_game()

AttributeError: 'NoneType' object has no attribute 'draw'